# Task 11 — Tests

**Wave 3** · target file: `backend/tests/test_api.py` · prerequisites: task 10

**🎯 Goal:** TestClient API tests.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [1]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

repo root: /Users/mario/code/Macrocosm
models: ['fake_image_model.pkl']


## What to build
Fill `backend/tests/test_api.py`. Build **tiny fixtures** (a small fake baseline dict
`{'model','features'}` + the fake image model) and point `BASELINE_PATH`/`IMAGE_MODEL_PATH` at them
**before** importing `backend.main`. Tests:
- `GET /` -> 200, `status == "ok"`,
- `POST /predict` image-only -> 200, has `z` + `distance_gly`,
- `POST /predict` image + tabular JSON -> 200,
- `POST /predict` a `32x32x5` cutout -> 422.

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [2]:
import io, json, tempfile
import joblib
import numpy as np
from sklearn.linear_model import LinearRegression
from fastapi.testclient import TestClient
from backend.fake_model import RandomRedshiftModel

# --- tiny fake model fixtures ---
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30) * 0.4),
             "features": list(range(16))}, f"{TMP}/b.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/i.pkl")

# Override settings paths BEFORE the app loads models
import backend.config, backend.model
backend.config.settings.BASELINE_PATH = f"{TMP}/b.pkl"
backend.config.settings.IMAGE_MODEL_PATH = f"{TMP}/i.pkl"
backend.model._baseline = None   # reset cache so lifespan reloads
backend.model._image = None

import backend.main

def _npy(shape):
    buf = io.BytesIO()
    np.save(buf, np.random.rand(*shape).astype(np.float32))
    buf.seek(0)
    return buf.read()

with TestClient(backend.main.app) as client:
    # GET /
    r = client.get("/")
    assert r.status_code == 200 and r.json()["status"] == "ok", r.text

    # POST /predict image-only
    r = client.post("/predict",
                    data={"ra": "180.0", "dec": "0.0"},
                    files={"file": ("c.npy", _npy((64,64,5)), "application/octet-stream")})
    assert r.status_code == 200, r.text
    assert "z" in r.json() and "distance_gly" in r.json()

    # POST /predict image + tabular
    tab = {"dered_u":19.5,"dered_g":18.0,"dered_r":17.5,"dered_i":17.0,"dered_z":16.8,
           "expRad_r":2.0,"deVRad_r":1.5,"petroRad_r":3.0,"petroR50_r":1.2,"petroR90_r":3.0,
           "fracDeV_r":0.5}
    r = client.post("/predict",
                    data={"ra": "180.0", "dec": "0.0", "tabular": json.dumps(tab)},
                    files={"file": ("c.npy", _npy((64,64,5)), "application/octet-stream")})
    assert r.status_code == 200, r.text
    assert "z" in r.json()

    # POST /predict wrong shape -> 422
    r = client.post("/predict",
                    data={"ra": "180.0", "dec": "0.0"},
                    files={"file": ("c.npy", _npy((32,32,5)), "application/octet-stream")})
    assert r.status_code == 422, r.text

print("All assertions passed!")

All assertions passed!


## ➡️ Move it into `backend/tests/test_api.py`
Once the cell above passes, put your implementation into **`backend/tests/test_api.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [3]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "backend/tests/", "-q"], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-400:])
assert r.returncode == 0, "pytest failed"
print("tests OK")

....                                                                     [100%]
4 passed in 2.71s


tests OK


> 💬 *Discuss freely with the team; post anything useful to the Knowledge Base.*

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/11-tests
!git add backend/tests/test_api.py notebooks/tasks-2026-6-17/11-tests/task.ipynb
!git commit -m "11-tests: Tests"
!git push -u origin task/11-tests
# then merge back into 2026.6.17 -> see MERGE.md in this folder